# Rung 40 — ARM B: the connector trains, at 4e-5

**THE ONE VARIABLE (two flags, one idea): the connector receives gradient.**

`modules_to_save` = the two merger layers, by **full path**, and `connector_lr = 4e-5` — 1/5 of
the LoRA LR, following `FICHAS.md:361`. The flags move together because a connector LR is
meaningless without the connector being trainable, exactly as rung 39 declared its own two-flag
single variable.

🔴 **Known, declared, measured deviation**: `save_pretrained_merged` carries the trained *weight*
and **drops the trained bias** (G3, 2026-08-13). Path B (LoRA on the connector) would not fix it —
LoRA never adapts biases either. If this arm comes back null, the dropped bias is a declared
candidate explanation.


## 1 — Parameters (papermill overrides this cell)

In [ ]:
SMOKE = True   # True -> Qwen3.5-2B + synthetic noise. False -> the 27B on rung 18's data.

# 🔴 WHERE is a SEPARATE axis from WHAT. These used to be welded together — SMOKE
# also meant "UNAM paths" — which would have sent a POD rehearsal writing to
# /data/uaq_user, a directory that does not exist there. A rehearsal is the small
# model on the real machine, so the machine has to be nameable on its own.
# "" keeps the old UNAM-vs-pod defaults for interactive use; the chain passes both.
WORK_DIR = ""
HF_HOME  = ""

# 5 steps is enough to check a path, but a rehearsal that finishes in 90 s never
# gives the watchdog a chance to misfire, and misfiring is the thing we are testing.
# The chain raises this so training outlasts several watchdog checks.
SMOKE_STEPS = 5


## 2 — Environment. Fail here, not after loading a 27B.

In [ ]:
import unsloth  # noqa: F401  MUST precede transformers
import importlib.metadata as md, torch

for pkg, want in {"unsloth": "2026.8.15", "unsloth_zoo": "2026.8.10"}.items():
    got = md.version(pkg)
    assert got == want, f"{pkg} is {got}, expected {want} — the gates were measured on {want}"
assert torch.cuda.is_available(), "no CUDA"
print("gpu     ", torch.cuda.get_device_name(0))
print("free GiB", round(torch.cuda.mem_get_info()[0] / 2**30, 1))
print("SMOKE   ", SMOKE)


## 3 — Config. Inline, per the repo spec.

In [ ]:
import logging, sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
for d in ("_models", "_tools"):
    p = str(Path.cwd() / d)
    if p not in sys.path:
        sys.path.insert(0, p)

from gen36_arm import ArmConfig, main  # noqa: E402

cfg = ArmConfig(
    arm='B_connector',
    modules_to_save=(
        "model.visual.merger.linear_fc1",
        "model.visual.merger.linear_fc2",
    ),
    connector_lr=4e-5,
    smoke=SMOKE,
    smoke_steps=SMOKE_STEPS,
    work_dir=WORK_DIR or ("/data/uaq_user/tmp/leo_arm40" if SMOKE
                          else "/workspace/repo_leo/experiments/40-gen36-recipe-connector/runs"),
    hf_home=HF_HOME or ("/data/uaq_user/hf_cache" if SMOKE else ""),
)
cfg


## 4 — Run. Guards raise; papermill turns that into a non-zero exit.

In [ ]:
result = main(cfg)

## 5 — What moved, and what the guards saw.

In [ ]:
import json
print("VERDICT  :", result["verdict"])
print("diff vs 38:", json.dumps(result["diff_vs_rung38"], default=str))
print("coverage :", {k: v for k, v in result["coverage"].items() if k != "wrapped"})
print("recorded :", result["recorded_fields"])
print("loss     :", result.get("train_loss"))
print("peak VRAM:", round(result.get("peak_vram_gib", 0), 2), "GiB")
if "param_groups_post" in result:
    print("groups   :", result["param_groups_post"]["groups"])
if "merge_check" in result:
    print("merge    :", result["merge_check"].get("known_bias_loss", "clean"))
print("control  :", result["control_run"], result["control_scores"])
